In [11]:
from arcgis.gis import GIS
import json

# Log in
gis = GIS("home")


# 🔍 Get web map by item ID
webmap_item = gis.content.get("e0fbb9b7108b4166a4e5fcdccb2390d8")  # Replace with actual item ID
webmap_data = webmap_item.get_data()

"""
results = gis.content.search(
    query='title:"Central Florida Wetland Evaluation Tool" AND owner:danielle.ivey@audubon.org_audubon',
    item_type="Web Map"
)

if not results:
    raise Exception("❌ Web map not found.")

webmap_item = results[0]
webmap_data = webmap_item.get_data()
"""

danielle = gis.users.get("danielle.ivey@audubon.org_audubon")  # Use her exact username

# Get all items from the user's content
all_items = danielle.items(max_items=600)

# Filter only hosted feature layers
replacement_layers = [item for item in all_items if item.type == "Feature Service"]




# Build lookup of title → ID
lookup = {item.title: item.id for item in replacement_layers}

print("\n📦 Titles from replacement layers:")
for k in lookup.keys():
    print(f"- '{k}'")

# 🔁 Recursive function to replace layers, including inside group layers
def replace_layers(layers, lookup):
    replaced = 0
    for layer in layers:
        title = layer.get("title")

        print(f"🗂 Web map layer title: '{title}'")
        
        if title in lookup:
            print(f"✅ Replacing: {title}")
            layer["itemId"] = lookup[title]
            if "url" in layer:
                del layer["url"]
            replaced += 1
        else:
            print(f"❌ No match found for: {title}")

        # If group layer, recursively process sublayers
        if "layers" in layer:
            print(f"🔽 Entering group layer: {title}")
            replaced += replace_layers(layer["layers"], lookup)

    return replaced

# 🔁 Run the replacement
total_replaced = replace_layers(webmap_data["operationalLayers"], lookup)

# 📝 Save back to AGOL
if total_replaced > 0:
    webmap_item.update(item_properties={"text": json.dumps(webmap_data)})
    success = webmap_item.update(item_properties={"text": json.dumps(webmap_data)})
    print("✅ Update success:", success)
    print(f"\n✅ {total_replaced} layers replaced and web map updated!")
else:
    print("\n⚠️ No layers replaced. Double-check layer titles and group nesting.")


📦 Titles from replacement layers:
- 'Florida_Water_Management_Districts'
- 'Mitigation_Bank_Watersheds (1)'
- 'Potmap_wells_May2010'
- 'Water_Use_Projected'
- 'FL_wildlife_Corridor_2021'
- 'Critical_Wildlife_Areas_Florida'
- 'Outstanding_Florida_Waters'
- 'flma_202310 (1)'
- 'PSAB_CFWI_16AUG2018'
- 'aerial_combin_index'
- 'OsceolaTaxParcels'
- 'lulc2020'
- 'SFWMD_Land_Cover_Land_Use_2017_2019'
- 'Wet places'
- 'RFLPP_1_WFL1'
- 'Enriched Block Group Renters_danielleivey'
- 'Summarized Streams_danielleivey'
- 'Counties in the state of Florida'
- 'Suitability 59_60_test'
- 'Sub_Watersheds'
- 'Wetland Map 3_WFL1'
- 'FL_Wetlands_Geopackage'
- 'Map2_WFL1'
- 'Florida_County_Boundaries_with_FDOT_Districts__8977087119664436883'
- 'Strategic_Planning_Basins'
- 'CFWI Water Management Suitability_WFL1'
- 'RFLPP_WET_53_60_acquired'
- 'Counties in the state of Florida view'
- 'FEGN 2021'
- 'Area 50 to 200 Acres Storage 58'
- 'Area 50 to 200 Acres Recharge 54'
- 'Area 200 to 400 Acres Storage 50'
- 